# **Loading the Data**

In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/gdrive')

nhanes = pd.read_csv("/content/gdrive/MyDrive/nhanescopy.csv")
nhanes

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDRETH1,DMQMILIZ,DMDBORN4,DMDYRUSR,DMDEDUC2,...,HIQ032F,HIQ032H,HIQ032I,HIQ210,INDFMMPC,INQ300,IND310,BPXOSY1,BPXODI1,LBXTC
0,132141.0,12.0,2.0,2.0,56.0,4.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,2.0,3.0,1.0,NaN,115.0,70.0,205.0
1,136977.0,12.0,2.0,1.0,29.0,3.0,2.0,2.0,2.0,5.0,...,NaN,NaN,NaN,2.0,3.0,2.0,2.0,125.0,71.0,193.0
2,134280.0,12.0,2.0,2.0,25.0,3.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,2.0,3.0,1.0,NaN,111.0,69.0,87.0
3,132522.0,12.0,2.0,2.0,64.0,3.0,2.0,1.0,NaN,4.0,...,NaN,NaN,NaN,2.0,NaN,NaN,NaN,113.0,90.0,215.0
4,138725.0,12.0,2.0,2.0,28.0,3.0,2.0,1.0,NaN,5.0,...,NaN,NaN,NaN,2.0,3.0,1.0,NaN,114.0,69.0,215.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5698,138255.0,12.0,2.0,2.0,65.0,5.0,2.0,1.0,NaN,4.0,...,NaN,8.0,NaN,2.0,1.0,2.0,1.0,99.0,62.0,144.0
5699,134584.0,12.0,2.0,1.0,56.0,1.0,2.0,1.0,NaN,4.0,...,NaN,NaN,NaN,2.0,3.0,2.0,3.0,137.0,102.0,235.0
5700,133659.0,12.0,2.0,2.0,80.0,3.0,2.0,1.0,NaN,4.0,...,NaN,NaN,9.0,2.0,NaN,NaN,NaN,126.0,65.0,156.0
5701,131701.0,12.0,2.0,2.0,46.0,3.0,2.0,1.0,NaN,3.0,...,NaN,NaN,NaN,1.0,2.0,1.0,NaN,121.0,76.0,291.0


In [ ]:
bp_purple = nhanes.copy().dropna(subset=['BPXOSY1', 'BPXODI1', 'HIQ032I'])
chol_purple = nhanes.copy().dropna(subset=['LBXTC', 'HIQ032I'])

# **Data Splitting**

In [ ]:
def create_bp_col(row):
  if pd.isna(row["BPXOSY1"]) or pd.isna(row["BPXODI1"]):
        return None
  if row["BPXOSY1"] < 120 and row["BPXODI1"] < 80:
    return "normal"
  else:
    return "abnormal"

nhanes["BP"] = nhanes.apply(create_bp_col, axis=1)


def create_chol_col(val):
  if pd.isna(val):
        return None
  if val < 200:
    return "normal"
  if val <= 239:
    return "borderline"
  return "high"

nhanes["CHOL"] = nhanes["LBXTC"].apply(create_chol_col)

nhanes = nhanes.drop(columns=["BPXOSY1","BPXODI1","LBXTC"])
nhanes[["BP","CHOL"]].head()

,BP,CHOL
0,normal,borderline
1,abnormal,normal
2,normal,normal
3,abnormal,borderline
4,normal,borderline


In [ ]:
from sklearn.model_selection import train_test_split

bp_purple = nhanes.copy().dropna(subset=['BP'])
chol_purple = nhanes.copy().dropna(subset=['CHOL'])

bp_training_purple, bp_test_purple = train_test_split(bp_purple, test_size=0.2, random_state=42)
chol_training_purple, chol_test_purple = train_test_split(chol_purple, test_size=0.2, random_state=42)

# **Pre processing**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier


In [ ]:
from joblib.pool import TemporaryResourcesManager
from sklearn.pipeline import Pipeline,  make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
import numpy as np
from sklearn.decomposition import PCA, FastICA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.model_selection import GridSearchCV, cross_validate, RandomizedSearchCV

binary_features = ['RIAGENDR','DMQMILIZ','DMDBORN4','ALQ111','ALQ151','HIQ011','INQ300'] #gender, active military, country of birth, ever had alc, daily 4-5+ drinks, covered by health insurance, savings>5000
# binary_features = ['RIAGENDR', 'DMQMILIZ', 'DMDBORN4', 'ALQ111', 'ALQ151', 'HIQ011', 'INQ300', 'is_elderly', 'heavy_drinker', 'binge_drinker', 'large_household', 'living_alone', 'immigrant', 'recent_immigrant', 'low_ses', 'ALQ121_missing', 'ALQ130_missing', 'ALQ170_missing', 'IND310_missing']
onehot_features = ['RIDRETH1','DMDMARTZ'] #hispanic/other race origin, marital status
ordinal_features = ['DMDYRUSR','DMDEDUC2','ALQ121','INDFMMPC','IND310'] #years in us, education age 20+, drinking frequency per 12 months, monthly poverty category, total savings category
numeric_features = ['RIDAGEYR','DMDHHSIZ','WTINT2YR','WTMEC2YR','ALQ130','ALQ170'] #age, household size, survey weight, survey weight, av drinks per day, 4/5+ drinks last 30 days
# numeric_features = ['RIDAGEYR', 'DMDHHSIZ', 'WTINT2YR', 'WTMEC2YR', 'ALQ130', 'ALQ170', 'RIDAGEYR_sq', 'alcohol_load', 'edu_income_score', 'insurance_count', 'age_gender']
insurance_features = ['HIQ032A','HIQ032B','HIQ032D','HIQ032F','HIQ032H','HIQ032I'] #private insurance, medicare, medicaid, military healthcare, state sponsored insurance, other insurance
id_features = ['SEQN'] #id
survey_features = ['SDDSRVYR', 'RIDSTATR'] #data release cycle, interview exam status


binary_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")) #imputes binary features with most_frequent value
])
onehot_transformer = Pipeline([  #imputes categorical features with more frequent value, then one hot encodes
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])
ordinal_transformer = Pipeline([ #imputes ordinal features with median value, then scales
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())   # EDIT: scaling added
])
numeric_transformer = Pipeline([ #imputes numeric features with median value, then scales
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())   # EDIT: scaling added
])
insurance_transformer = Pipeline([ #imputes insurance features with most frequent value, then one hot encodes
    ("imputer", SimpleImputer(strategy="most_frequent")),
     ("encoder", OneHotEncoder(handle_unknown="ignore"))
     ])
survey_transformer = Pipeline([ #imputes survey features with most frequent value
    ("imputer", SimpleImputer(strategy="most_frequent"))
])
id_transformer = Pipeline([ #imputes id feature with most frequent value
    ("imputer", SimpleImputer(strategy="most_frequent"))
])
chol_transformer =Pipeline([  #imputes categorical features with more frequent value, then one hot encodes
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
bp_transformer =Pipeline([  #imputes categorical features with more frequent value, then one hot encodes
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])


def clean_data(X):

    X = pd.DataFrame(X).copy()

    replace_vals = [7, 9, 77, 99, 777, 999]
    X = X.replace(replace_vals, np.nan)

    if 'ALQ121' in X.columns:
        X.loc[X['ALQ121'] < 1, 'ALQ121'] = np.nan

    if 'ALQ170' in X.columns:
        X.loc[X['ALQ170'] < 0.1, 'ALQ170'] = 0

    binary_map = {1.0: 1.0, 2.0: 0.0}
    binary_cols = ['RIAGENDR', 'DMQMILIZ', 'DMDBORN4', 'ALQ111', 'ALQ151', 'HIQ011', 'INQ300']
    for col in binary_cols:
        if col in X.columns:
            X[col] = X[col].replace(binary_map)
    return X

    # # FEATURE ENGINEERING

    # # age features
    # X["RIDAGEYR_sq"] = X["RIDAGEYR"] ** 2
    # X["is_elderly"] = (X["RIDAGEYR"] >= 65).astype(float)

    # # alcohol features
    # X["alcohol_load"] = (X["ALQ121"] * X["ALQ130"])
    # X["heavy_drinker"] = (X["ALQ130"] >= 5).astype(float)
    # X["binge_drinker"] = (X["ALQ170"] >= 8).astype(float)

    # # household features
    # X["large_household"] = (X["DMDHHSIZ"] >= 5).astype(float)
    # X["living_alone"] = (X["DMDHHSIZ"] == 1).astype(float)

    # # immigrant features
    # X["immigrant"] = (X["DMDBORN4"] == 0).astype(float)
    # X["recent_immigrant"] = ((X["DMDBORN4"] == 0) &(X["DMDYRUSR"] <= 2)).astype(float)

    # # socioeconomic features
    # X["low_ses"] = ((X["INDFMMPC"] == 1) & (X["DMDEDUC2"] <= 2)).astype(float)
    # X["edu_income_score"] = (X["DMDEDUC2"] * X["INDFMMPC"])

    # # insurance features
    # insurance_cols = ['HIQ032A', 'HIQ032B', 'HIQ032D', 'HIQ032F', 'HIQ032H', 'HIQ032I']
    # X["insurance_count"] = (X[insurance_cols].notna().sum(axis=1))

    # # interaction features
    # X["age_gender"] = (X["RIDAGEYR"] * X["RIAGENDR"])
    # X["age_alcohol"] = (X["RIDAGEYR"] * X["alcohol_load"])
    # X["poverty_insurance"] = (X["INDFMMPC"] * X["insurance_count"])
    # X["poverty_gender"] = (X["INDFMMPC"] * X["RIAGENDR"])
    # X["poverty_age"] = (X["INDFMMPC"] * X["RIDAGEYR"])
    # X["edu_alcohol"] = (X["DMDEDUC2"] * X["ALQ130"])


    # # missing indicators
    # missing_cols = ['ALQ121', 'ALQ130', 'ALQ170', 'IND310']

    # for col in missing_cols:
    #     X[col + "_missing"] = ( X[col].isna()).astype(float)


preprocessor_bp = ColumnTransformer([
    ("binary", binary_transformer, binary_features),
    ("onehot", onehot_transformer, onehot_features),
    ("ordinal", ordinal_transformer, ordinal_features),
    ("numeric", numeric_transformer, numeric_features),
    ("insurance", insurance_transformer, insurance_features),
    ("survey", survey_transformer, survey_features),
    ("id", id_transformer, id_features),
     ("chol", chol_transformer, ["CHOL"])
], remainder="drop")

preprocessor_chol = ColumnTransformer([
    ("binary", binary_transformer, binary_features),
    ("onehot", onehot_transformer, onehot_features),
    ("ordinal", ordinal_transformer, ordinal_features),
    ("numeric", numeric_transformer, numeric_features),
    ("insurance", insurance_transformer, insurance_features),
    ("survey", survey_transformer, survey_features),
    ("id", id_transformer, id_features),
    ("bp", bp_transformer, ["BP"])
], remainder="drop")

plineb = Pipeline([
    ("clean", FunctionTransformer(clean_data)),   # EDIT
    ("prep", preprocessor_bp),
    ('ICA', FastICA(n_components=20, random_state=42))
])

plinec = Pipeline([
    ("clean", FunctionTransformer(clean_data)),   # EDIT
    ("prep", preprocessor_chol),
    ('ICA', FastICA(n_components=20, random_state=42))
    ])

X_train_bp = bp_training_purple.drop(columns=["BP"])
X_train_chol = chol_training_purple.drop(columns=["CHOL"])

y_train_bp = bp_training_purple["BP"]
y_train_chol = chol_training_purple["CHOL"]

# **Grid search + Random search**

In [ ]:


param_grid = {
    "stump": [DecisionTreeClassifier(max_depth=1, random_state=42, class_weight="balanced"),
              {
                  'model__criterion': ['gini', 'entropy'],
              }],

    "tree": [DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"),
             {
                 'model__max_depth': [3, 5, 7, 9, 11, None],
                 'model__criterion': ['gini', 'entropy'],
                 'model__min_samples_split': [2, 5, 10],
                 'model__min_samples_leaf': [1, 2, 4],
                 'model__max_features': ['sqrt', 'log2', None],
             }],

    "rf": [RandomForestClassifier(random_state=42, class_weight="balanced"),
           {
               'model__n_estimators': [100, 200, 500],
               'model__max_depth': [3, 5, 7, 9, 11, None],
               'model__criterion': ['gini', 'entropy'],
               'model__min_samples_split': [2, 5, 10],
               'model__min_samples_leaf': [1, 2, 4],
               'model__max_features': ['sqrt', 'log2', None]
           }],
    "ada": [AdaBoostClassifier(random_state=42),
            {
                'model__n_estimators': [50, 100, 200],
                'model__learning_rate': [0.01, 0.1, 1.0]
            }],
    "svm": [SVC(random_state=42, class_weight="balanced"),
            {
                'model__C': [0.1, 1, 10],
                'model__kernel': ['rbf', 'linear'],
                'model__gamma': ['scale', 'auto', 0.01]
            }],
    "nb": [GaussianNB(),
           {
               'model__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
           }],

    "logreg": [LogisticRegression(random_state=42, class_weight="balanced"),
    [
        {'model__penalty': ['l2'], 'model__solver': ['lbfgs', 'newton-cg', 'sag'], 'model__C': [0.01, 0.1, 1, 10, 100], 'model__max_iter': [100, 500, 1000]},
        {'model__penalty': ['l1'], 'model__solver': ['liblinear', 'saga'], 'model__C': [0.01, 0.1, 1, 10, 100],'model__max_iter': [100, 500, 1000]},
        {'model__penalty': ['elasticnet'], 'model__solver': ['saga'], 'model__C': [0.01, 0.1, 1, 10, 100], 'model__l1_ratio': [0.5],'model__max_iter': [100, 500, 1000]},

    ]
    ],
    "knn": [KNeighborsClassifier(),
            {
                'model__n_neighbors': [3, 5, 7, 9, 11, 15],
                'model__weights': ['uniform', 'distance'],
                'model__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
                'model__metric': ['euclidean', 'manhattan', 'chebyshev', 'minkowski']
            }],
    "mlp": [MLPClassifier(random_state=42),
            {
              'model__hidden_layer_sizes': [(50,), (100,), (100, 100)],
              'model__activation': ['relu', 'tanh'],
              'model__solver': ['adam'],
              'model__alpha': [0.0001, 0.001],
              'model__learning_rate': ['constant', 'adaptive'],
              'model__max_iter': [500, 1000, 1500]
            }],

    "gb": [GradientBoostingClassifier(random_state=42),
           {
                'model__n_estimators': [50, 100, 200],
                'model__learning_rate': [0.01, 0.1, 1.0],
                'model__max_depth': [3, 5, 7, 9, 11]
           }],
    "hgb": [HistGradientBoostingClassifier(random_state=42),
            {
                'model__max_iter': [100, 200, 500],
                'model__learning_rate': [0.01, 0.1, 1.0],
                'model__max_depth': [3, 5, 7, 9, 11],
                'model__l2_regularization': [0.0, 0.1, 0.5]
            }]
}


grid_models = ['logreg', 'nb', 'ada', 'stump', 'tree', 'knn', 'gb']
random_models = ['rf', 'svm', 'mlp', 'hgb']

In [ ]:
for name in param_grid:

    print(f"\n--- {name} ---")

    if name in grid_models:
        search = GridSearchCV(estimator=Pipeline([
    ('prep', plinec),
    ('model', param_grid[name][0])
]), param_grid=param_grid[name][1], cv=10, n_jobs=-1)
    else:
        search = RandomizedSearchCV(estimator=Pipeline([
    ('prep', plinec),
    ('model', param_grid[name][0])
]), param_distributions=param_grid[name][1], n_iter=40, cv=10, random_state=42, n_jobs=-1)

    search.fit(X_train_chol, y_train_chol)
    print(search.best_params_)
    print(search.best_score_)


--- stump ---


ValueError: Invalid parameter 'model' for estimator DecisionTreeClassifier(class_weight='balanced', max_depth=1, random_state=42). Valid parameters are: ['ccp_alpha', 'class_weight', 'criterion', 'max_depth', 'max_features', 'max_leaf_nodes', 'min_impurity_decrease', 'min_samples_leaf', 'min_samples_split', 'min_weight_fraction_leaf', 'monotonic_cst', 'random_state', 'splitter'].

In [ ]:
for name in param_grid:
    print(f"\n--- {name} ---")
    if name in grid_models:
        search = GridSearchCV(estimator=
    Pipeline([
    ('prep', plinec),
    ('model', param_grid[name][0])
]), param_grid=param_grid[name][1], cv=10, scoring = 'f1_macro', n_jobs=-1)
    else:
        search = RandomizedSearchCV(estimator=
    Pipeline([
    ('prep', plinec),
    ('model', param_grid[name][0])
]), param_distributions=param_grid[name][1], n_iter=40, cv=10, scoring = 'f1_macro', random_state=42, n_jobs=-1)
    search.fit(X_train_chol, y_train_chol)
    print(search.best_params_)
    print(search.best_score_)



# --- stump ---
# /usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
#   warnings.warn(
# {'model__criterion': 'gini'}
# 0.306121482305952

# --- tree ---
# /usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
#   warnings.warn(
# {'model__criterion': 'entropy', 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 2}
# 0.3683738591603889

# --- rf ---
# /usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
#   warnings.warn(
# {'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_features': 'log2', 'model__max_depth': 9, 'model__criterion': 'gini'}
# 0.39654220259913836

# --- ada ---
# /usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
#   warnings.warn(
# {'model__learning_rate': 1.0, 'model__n_estimators': 200}
# 0.2636823322925438

# --- svm ---
# /usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 18 is smaller than n_iter=40. Running 18 iterations. For exhaustive searches, use GridSearchCV.
#   warnings.warn(
# /usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
#   warnings.warn(
# {'model__kernel': 'rbf', 'model__gamma': 'scale', 'model__C': 1}
# 0.37911361645361763

# --- nb ---
# /usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
#   warnings.warn(
# {'model__var_smoothing': 1e-09}
# 0.3025784504556678

# --- logreg ---
# /usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
#   warnings.warn(
# {'model__C': 0.01, 'model__max_iter': 100, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
# 0.3592460418800676

# --- knn ---
# /usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [0.35458062 0.35918676 0.3609774  0.34478768 0.34116435 0.3286861
#  0.33924009 0.32913647 0.33043688 0.32740159 0.31880711 0.31382717
#         nan 0.3589767         nan 0.3527199         nan 0.33213761
#         nan 0.3297817         nan 0.33301869        nan 0.31265432
#         nan 0.3410497         nan 0.33964807        nan 0.3330969
#         nan 0.32890811        nan 0.31379706        nan 0.3008004
#  0.35458062 0.35918676 0.3609774  0.34478768 0.34116435 0.3286861
#  0.33924009 0.32913647 0.33043688 0.32740159 0.31880711 0.31382717
#  0.35458062 0.35918676 0.3609774  0.34478768 0.34116435 0.3286861
#  0.33924009 0.32913647 0.33043688 0.32740159 0.31880711 0.31382717
#  0.36381091 0.3589767  0.36244901 0.3527199  0.34104668 0.33213761
#  0.33609743 0.3297817  0.33587528 0.33301869 0.31869608 0.31265432
#  0.34204457 0.3410497  0.34165691 0.33964807 0.33308746 0.3330969
#  0.32629241 0.32890811 0.31695386 0.31379706 0.30436762 0.3008004
#  0.35458062 0.35918676 0.3609774  0.34478768 0.34116435 0.3286861
#  0.33924009 0.32913647 0.33043688 0.32740159 0.31880711 0.31382717
#  0.35458062 0.35918676 0.3609774  0.34478768 0.34116435 0.3286861
#  0.33924009 0.32913647 0.33043688 0.32740159 0.31880711 0.31382717
#  0.36381091 0.3589767  0.36244901 0.3527199  0.34104668 0.33213761
#  0.33609743 0.3297817  0.33587528 0.33301869 0.31869608 0.31265432
#  0.34204457 0.3410497  0.34165691 0.33964807 0.33308746 0.3330969
#  0.32629241 0.32890811 0.31695386 0.31379706 0.30436762 0.3008004
#  0.35458062 0.35918676 0.3609774  0.34478768 0.34116435 0.3286861
#  0.33924009 0.32913647 0.33043688 0.32740159 0.31880711 0.31382717
#  0.35458062 0.35918676 0.3609774  0.34478768 0.34116435 0.3286861
#  0.33924009 0.32913647 0.33043688 0.32740159 0.31880711 0.31382717
#         nan 0.3589767         nan 0.3527199         nan 0.33213761
#         nan 0.3297817         nan 0.33301869        nan 0.31265432
#         nan 0.3410497         nan 0.33964807        nan 0.3330969
#         nan 0.32890811        nan 0.31379706        nan 0.3008004
#  0.35458062 0.35918676 0.3609774  0.34478768 0.34116435 0.3286861
#  0.33924009 0.32913647 0.33043688 0.32740159 0.31880711 0.31382717]
#   warnings.warn(
# /usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
#   warnings.warn(
# {'model__algorithm': 'ball_tree', 'model__metric': 'manhattan', 'model__n_neighbors': 3, 'model__weights': 'uniform'}
# 0.36381091093023765

# --- mlp ---


--- stump ---


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


{'model__criterion': 'gini'}
0.306121482305952

--- tree ---


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


{'model__criterion': 'entropy', 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 2}
0.3683738591603889

--- rf ---


...

{'model__C': 0.01, 'model__max_iter': 100, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
0.3592460418800676

--- knn ---


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [0.35458062 0.35918676 0.3609774  0.34478768 0.34116435 0.3286861
 0.33924009 0.32913647 0.33043688 0.32740159 0.31880711 0.31382717
        nan 0.3589767         nan 0.3527199         nan 0.33213761
        nan 0.3297817         nan 0.33301869        nan 0.31265432
        nan 0.3410497         nan 0.33964807        nan 0.3330969
        nan 0.32890811        nan 0.31379706        nan 0.3008004
 0.35458062 0.35918676 0.3609774  0.34478768 0.34116435 0.3286861
 0.33924009 0.32913647 0.33043688 0.32740159 0.31880711 0.31382717
 0.35458062 0.35918676 0.3609774  0.34478768 0.34116435 0.3286861
 0.33924009 0.32913647 0.33043688 0.32740159 0.31880711 0.31382717
 0.36381091 0.3589767  0.36244901 0.3527199  0.34104668 0.33213761
 0.33609743 0.3297817  0.33587528 0.33301869 0.31869608 0.31265432
 0.34204457 0.3410497  0.34165691 0.33964807 0.33308746 0.33

{'model__algorithm': 'ball_tree', 'model__metric': 'manhattan', 'model__n_neighbors': 3, 'model__weights': 'uniform'}
0.36381091093023765

--- mlp ---


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


{'model__solver': 'adam', 'model__max_iter': 1000, 'model__learning_rate': 'adaptive', 'model__hidden_layer_sizes': (100, 100), 'model__alpha': 0.001, 'model__activation': 'relu'}
0.3803372779418871

--- gb ---


# **05/08 cross validation after accuracy chol param**

In [ ]:
#original params

models = {
    "stump": DecisionTreeClassifier(max_depth=1, random_state=42, class_weight="balanced"),
    "tree": DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced"),
    "rf": RandomForestClassifier(random_state=42, class_weight="balanced"),
    "ada": AdaBoostClassifier(random_state=42),
    "svm": SVC(random_state=42, class_weight="balanced"),
    "nb": GaussianNB(),
    "logreg": LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"),
    "knn": KNeighborsClassifier(),
    "mlp": MLPClassifier(random_state=42, max_iter=1000),
    "gb": GradientBoostingClassifier(random_state=42),
    "hgb": HistGradientBoostingClassifier(random_state=42),

    "voting_clf": VotingClassifier(estimators=[
    ('stump', DecisionTreeClassifier(max_depth=1, random_state=42, class_weight="balanced")),
    ('ada', AdaBoostClassifier(random_state=42)),
    ('tree', DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced")),
    ('rfc', RandomForestClassifier(random_state=42, class_weight="balanced")),
    ('nb', GaussianNB()),
    ('lr', LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")),
    ('knn', KNeighborsClassifier()),
    ('svm', SVC(random_state=42, class_weight="balanced")),
     ('mlp', MLPClassifier(random_state=42, max_iter=1000)),
     ('gb', GradientBoostingClassifier(random_state=42)),
     ('hgb', HistGradientBoostingClassifier(random_state=42)),
    ], voting='hard'),

    "voting_clf_soft": VotingClassifier(estimators=[
        ('ada', AdaBoostClassifier(random_state=42)),
        ('rfc', RandomForestClassifier(random_state=42, class_weight="balanced")),
        ('nb', GaussianNB()),
        ('lr', LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")),
        ('svm', SVC(random_state=42, class_weight="balanced", probability=True)),
        ('gb', GradientBoostingClassifier(random_state=42)),
        ('hgb', HistGradientBoostingClassifier(random_state=42)),
    ], voting='soft'),

    "stacking_clf": StackingClassifier(
        estimators=[
            ('ada', AdaBoostClassifier(random_state=42)),
            ('tree', DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced")),
            ('rfc', RandomForestClassifier(random_state=42, class_weight="balanced")),
            ('nb', GaussianNB()),
            ('lr', LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")),
            ('svm', SVC(random_state=42, class_weight="balanced", probability=True)),
            ('gb', GradientBoostingClassifier(random_state=42)),
            ('hgb', HistGradientBoostingClassifier(random_state=42)),
        ],
        final_estimator=LogisticRegression(
            random_state=42,
            max_iter=1000
        )
    )
}




--- BP: stump ---
Feature Importance for stump:
ratio [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
features [ 4 19 17 18 16 15 13 14 11 10  9 12  8  7  6  5  3  2  1  0]
{'fit_time': array([0.08116889, 0.05690718, 0.04131198, 0.03703475, 0.03621602,
       0.03638029, 0.03746724, 0.03611231, 0.04483271, 0.06544757]), 'score_time': array([0.02796412, 0.02876902, 0.02357507, 0.03881001, 0.02331734,
       0.02528477, 0.02395296, 0.02240849, 0.05500865, 0.02309847]), 'test_accuracy': array([0.62358277, 0.60770975, 0.64172336, 0.61904762, 0.6553288 ,
       0.64399093, 0.6462585 , 0.63038549, 0.60454545, 0.65      ]), 'test_precision_macro': array([0.61485619, 0.59710843, 0.63443834, 0.61210471, 0.65430588,
       0.63592923, 0.64076421, 0.62169056, 0.5987341 , 0.64343821]), 'test_recall_macro': array([0.61239791, 0.5928377 , 0.63334031, 0.61210471, 0.65707853,
       0.63225131, 0.62374869, 0.61839791, 0.59938182, 0.63161547]), 'test_f1_macro': array([0.61297792, 0.59294

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m


--- CHOL: stump ---
Feature Importance for stump:
ratio [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
features [ 9 19 17 18 16 15 13 14 12 11 10  8  7  6  5  4  3  2  1  0]
{'fit_time': array([0.07845736, 0.06312895, 0.04847741, 0.07376218, 0.0566113 ,
       0.05057049, 0.09505844, 0.08038521, 0.05293179, 0.05339646]), 'score_time': array([0.02577639, 0.03854799, 0.0245223 , 0.02313733, 0.05121779,
       0.0334239 , 0.05172086, 0.02273607, 0.02012944, 0.03642297]), 'test_accuracy': array([0.51941748, 0.47572816, 0.51699029, 0.53398058, 0.51213592,
       0.45873786, 0.50121655, 0.50851582, 0.51581509, 0.49635036]), 'test_precision_macro': array([0.28883053, 0.27497606, 0.29149476, 0.29694307, 0.28234994,
       0.26250881, 0.27853989, 0.29293598, 0.28996585, 0.28560027]), 'test_recall_macro': array([0.40246894, 0.37991255, 0.42457207, 0.42750493, 0.38703141,
       0.35635965, 0.38076574, 0.42042018, 0.41252509, 0.40829653]), 'test_f1_macro': array([0.31867549, 0.296

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m


--- CHOL: ada ---
Feature Importance for ada:
ratio [0.         0.         0.03700192 0.02735788 0.         0.
 0.13767592 0.07577454 0.         0.51769221 0.         0.04823749
 0.         0.         0.         0.         0.         0.
 0.10205505 0.054205  ]
features [ 9  6 18  7 19 11  2  3 16 17 15 14  8 10 13 12  4  5  1  0]
{'fit_time': array([1.02248168, 1.02120209, 1.17714787, 1.0245707 , 1.01562262,
       1.01411057, 1.13552833, 1.49643683, 1.50688434, 1.39547086]), 'score_time': array([0.02558255, 0.02526546, 0.0249908 , 0.02509952, 0.02581978,
       0.02581072, 0.04018855, 0.04358888, 0.03992677, 0.02540541]), 'test_accuracy': array([0.64563107, 0.64563107, 0.6407767 , 0.64563107, 0.64563107,
       0.64563107, 0.64720195, 0.64476886, 0.64476886, 0.64476886]), 'test_precision_macro': array([0.21521036, 0.21521036, 0.21463415, 0.21521036, 0.21521036,
       0.21521036, 0.21573398, 0.21492295, 0.21492295, 0.21492295]), 'test_recall_macro': array([0.33333333, 0.33333333, 0.3

...

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perce

{'fit_time': array([25.72866702, 26.57362223, 26.09188485, 25.55702615, 20.51862383,
       21.96744776, 26.46845889, 26.45372891, 26.73638844, 26.47299504]), 'score_time': array([0.17160106, 0.16813421, 0.33238554, 0.28487468, 0.1657145 ,
       0.17514181, 0.17119122, 0.17589855, 0.16922164, 0.28146505]), 'test_accuracy': array([0.64852608, 0.6553288 , 0.67120181, 0.66666667, 0.6893424 ,
       0.68480726, 0.65759637, 0.65986395, 0.65681818, 0.66590909]), 'test_precision_macro': array([0.64161512, 0.65126971, 0.669967  , 0.66060965, 0.68405257,
       0.68154708, 0.651172  , 0.65408769, 0.65469223, 0.66253754]), 'test_recall_macro': array([0.6294555 , 0.63298429, 0.6488377 , 0.65039791, 0.67595812,
       0.66701571, 0.6399267 , 0.6413089 , 0.63337118, 0.64567169]), 'test_f1_macro': array([0.62947404, 0.63201581, 0.64853218, 0.65149959, 0.67764706,
       0.66843361, 0.64056352, 0.64181287, 0.63165627, 0.64573463])}


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(



--- CHOL: voting_clf ---
{'fit_time': array([35.55523181, 36.25021243, 36.09455705, 35.44431114, 35.24913669,
       35.39048767, 36.17309451, 35.98734426, 37.43044758, 32.93358254]), 'score_time': array([0.18452215, 0.35818362, 0.19072819, 0.28883219, 0.20055509,
       0.21841669, 0.18220377, 0.18090439, 0.18612218, 0.18380737]), 'test_accuracy': array([0.63349515, 0.6407767 , 0.63592233, 0.62378641, 0.6407767 ,
       0.61650485, 0.61800487, 0.62773723, 0.6350365 , 0.60827251]), 'test_precision_macro': array([0.44120724, 0.42586335, 0.4299885 , 0.32933455, 0.44298475,
       0.40530466, 0.38475177, 0.41818182, 0.40769147, 0.31384468]), 'test_recall_macro': array([0.37428274, 0.3673691 , 0.36647416, 0.33423582, 0.36847788,
       0.35395035, 0.34655408, 0.37008003, 0.35635061, 0.33246151]), 'test_f1_macro': array([0.34976378, 0.3440577 , 0.34179117, 0.28693415, 0.33691133,
       0.32689134, 0.31462538, 0.34613144, 0.32317188, 0.29267133])}

--- BP: voting_clf_soft ---
{'fit_time': 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#accuracy based params
models = {
    "stump": DecisionTreeClassifier(criterion= 'gini', max_depth=1, random_state=42, class_weight="balanced"),
    "tree": DecisionTreeClassifier(criterion= 'gini', max_depth=None, max_features= 'sqrt', min_samples_leaf=1, min_samples_split=2, random_state=42, class_weight="balanced"),
    "rf": RandomForestClassifier(criterion='entropy', max_depth= None, max_features= 'log2', min_samples_leaf=1, min_samples_split=5, n_estimators=500, random_state=42, class_weight="balanced"),
    "ada": AdaBoostClassifier(n_estimators=200, learning_rate = 1.0, random_state=42),
    "svm": SVC(kernel='rbf', gamma='scale', C=10, random_state=42, class_weight="balanced"),
    "nb": GaussianNB(var_smoothing= 1e-09),
    "logreg": LogisticRegression(C= 0.01, max_iter= 100, penalty= 'elasticnet', solver='saga', l1_ratio=0.5,  random_state=42, class_weight="balanced"),
    "knn": KNeighborsClassifier(algorithm= 'auto', metric='chebyshev', n_neighbors= 15, weights= 'distance'),
    "mlp": MLPClassifier(solver='adam', max_iter= 500, learning_rate= 'constant', hidden_layer_sizes= (50,),alpha= 0.0001, activation='relu', random_state=42),
    "gb": GradientBoostingClassifier(learning_rate= 0.01, max_depth= 3, n_estimators= 200, random_state=42),
    "hgb": HistGradientBoostingClassifier(max_iter= 100, max_depth= 3, learning_rate= 0.01, l2_regularization= 0.1, random_state=42),

    "voting_clf": VotingClassifier(estimators=[
    ('stump', DecisionTreeClassifier(criterion= 'gini', max_depth=1, random_state=42, class_weight="balanced")),
    ('ada', AdaBoostClassifier(n_estimators=200, learning_rate = 1.0, random_state=42)),
    ('tree', DecisionTreeClassifier(criterion= 'gini', max_depth=None, max_features= 'sqrt', min_samples_leaf=1, min_samples_split=2, random_state=42, class_weight="balanced")),
    ('rfc',RandomForestClassifier(criterion='entropy', max_depth= None, max_features= 'log2', min_samples_leaf=1, min_samples_split=5, n_estimators=500, random_state=42, class_weight="balanced")),
    ('nb', GaussianNB(var_smoothing= 1e-09)),
    ('lr', LogisticRegression(C= 0.01, max_iter= 100, penalty= 'elasticnet', solver='saga', l1_ratio=0.5,  random_state=42, class_weight="balanced")),
    ('knn', KNeighborsClassifier(algorithm= 'auto', metric='chebyshev', n_neighbors= 15, weights= 'distance')),
    ('svm', SVC(kernel='rbf', gamma='scale', C=10, random_state=42, class_weight="balanced")),
     ('mlp', MLPClassifier(solver='adam', max_iter= 500, learning_rate= 'constant', hidden_layer_sizes= (50,),alpha= 0.0001, activation='relu', random_state=42)),
     ('gb', GradientBoostingClassifier(learning_rate= 0.01, max_depth= 3, n_estimators= 200, random_state=42)),
     ('hgb', HistGradientBoostingClassifier(max_iter= 100, max_depth= 3, learning_rate= 0.01, l2_regularization= 0.1, random_state=42)),
    ], voting='hard'),

    "voting_clf_soft": VotingClassifier(estimators=[
        ('ada', AdaBoostClassifier(n_estimators=200, learning_rate = 1.0, random_state=42)),
        ('rfc',RandomForestClassifier(criterion='entropy', max_depth= None, max_features= 'log2', min_samples_leaf=1, min_samples_split=5, n_estimators=500, random_state=42, class_weight="balanced")),
        ('nb', GaussianNB(var_smoothing= 1e-09)),
        ('lr', LogisticRegression(C= 0.01, max_iter= 100, penalty= 'elasticnet', solver='saga', l1_ratio=0.5,  random_state=42, class_weight="balanced")),
        ('svm', SVC(kernel='rbf', gamma='scale', C=10, random_state=42, probability=True, class_weight="balanced")),
        ('gb', GradientBoostingClassifier(learning_rate= 0.01, max_depth= 3, n_estimators= 200, random_state=42)),
        ('hgb', HistGradientBoostingClassifier(max_iter= 100, max_depth= 3, learning_rate= 0.01, l2_regularization= 0.1, random_state=42)),
    ], voting='soft'),

    "stacking_clf": StackingClassifier(
        estimators=[
            ('ada', AdaBoostClassifier(n_estimators=200, learning_rate = 1.0, random_state=42)),
            ('tree', DecisionTreeClassifier(criterion= 'gini', max_depth=None, max_features= 'sqrt', min_samples_leaf=1, min_samples_split=2, random_state=42, class_weight="balanced")),
            ('rfc', RandomForestClassifier(criterion='entropy', max_depth= None, max_features= 'log2', min_samples_leaf=1, min_samples_split=5, n_estimators=500, random_state=42, class_weight="balanced")),
            ('nb', GaussianNB(var_smoothing= 1e-09)),
            ('lr', LogisticRegression(C= 0.01, max_iter= 100, penalty= 'elasticnet', solver='saga', l1_ratio=0.5,  random_state=42, class_weight="balanced")),
            ('svm', SVC(kernel='rbf', gamma='scale', C=10, random_state=42, class_weight="balanced")),
            ('gb', GradientBoostingClassifier(learning_rate= 0.01, max_depth= 3, n_estimators= 200, random_state=42)),
            ('hgb', HistGradientBoostingClassifier(max_iter= 100, max_depth= 3, learning_rate= 0.01, l2_regularization= 0.1, random_state=42)),
        ],
        final_estimator=LogisticRegression(C= 0.01, max_iter= 100, penalty= 'elasticnet', solver='saga', l1_ratio=0.5,  random_state=42, class_weight="balanced")
    )
}

In [ ]:
scoring_list = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

In [ ]:
chol_results = {}
# Conduct 10-fold cross validation
for name in models:
    chol_cv = cross_validate(make_pipeline(plinec,models[name]), chol_training_purple.drop(columns=['CHOL']), chol_training_purple['CHOL'], cv=10, scoring=scoring_list)
    print(f"\n--- CHOL: {name} ---")
    chol_results[name] = chol_cv
    print(chol_results[name])

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for impu


--- CHOL: stump ---
{'fit_time': array([0.39904666, 0.2764802 , 0.22615838, 0.45919275, 0.27207541,
       0.24780178, 0.29611063, 0.3194859 , 0.22874069, 0.56028628]), 'score_time': array([0.05872154, 0.06555963, 0.05639553, 0.05586624, 0.05595541,
       0.05586338, 0.05586815, 0.05685592, 0.0555172 , 0.07578301]), 'test_accuracy': array([0.48786408, 0.4684466 , 0.48786408, 0.52427184, 0.51699029,
       0.45145631, 0.52554745, 0.49878345, 0.50121655, 0.45985401]), 'test_precision_macro': array([0.28314392, 0.27655502, 0.28559177, 0.29356337, 0.27431778,
       0.25916907, 0.27644757, 0.29037205, 0.28974009, 0.27518127]), 'test_recall_macro': array([0.39785634, 0.38199221, 0.41537354, 0.4224924 , 0.36618141,
       0.35260025, 0.37577988, 0.41538873, 0.41081226, 0.38942861]), 'test_f1_macro': array([0.30690728, 0.29509483, 0.31287243, 0.32833722, 0.30233766,
       0.27929057, 0.3088483 , 0.31687508, 0.31612055, 0.29453092])}


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/p


--- CHOL: tree ---
{'fit_time': array([1.2207284 , 0.97181296, 0.91676545, 1.27616644, 0.46043611,
       0.383744  , 0.30027008, 0.32406402, 0.35443258, 0.4199543 ]), 'score_time': array([0.05989552, 0.05841708, 0.08408618, 0.0909133 , 0.07750463,
       0.05540609, 0.05523658, 0.05776906, 0.05563664, 0.05623269]), 'test_accuracy': array([0.51213592, 0.5461165 , 0.52912621, 0.48543689, 0.49029126,
       0.51941748, 0.50364964, 0.53041363, 0.45012165, 0.52068127]), 'test_precision_macro': array([0.34417901, 0.39910064, 0.37687842, 0.32547112, 0.33312867,
       0.36420343, 0.35803003, 0.3765282 , 0.30228145, 0.3581993 ]), 'test_recall_macro': array([0.3438367 , 0.39842514, 0.37485612, 0.32371063, 0.3335636 ,
       0.36032044, 0.35673149, 0.37523313, 0.30033373, 0.35790313]), 'test_f1_macro': array([0.34338726, 0.39854894, 0.37567312, 0.32405357, 0.33308122,
       0.36153333, 0.35708185, 0.37514143, 0.30108692, 0.35587715])}


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/p

...


--- CHOL: voting_clf ---
{'fit_time': array([60.29609871, 60.64752388, 60.21778178, 60.54565382, 60.05497813,
       59.79897404, 59.78137279, 61.50180888, 61.25033355, 60.63058019]), 'score_time': array([0.38048744, 0.36855149, 0.38466167, 0.37714291, 0.36562419,
       0.36980605, 0.46632051, 0.55685663, 0.55197096, 0.3649354 ]), 'test_accuracy': array([0.65533981, 0.64563107, 0.64320388, 0.65048544, 0.65048544,
       0.63106796, 0.64963504, 0.63990268, 0.6350365 , 0.63017032]), 'test_precision_macro': array([0.45620223, 0.38448845, 0.32732733, 0.41756979, 0.58395062,
       0.37211083, 0.35090312, 0.26432707, 0.21375921, 0.21316872]), 'test_recall_macro': array([0.36754119, 0.34501146, 0.33419407, 0.34218121, 0.35547067,
       0.33934539, 0.3462646 , 0.33665195, 0.32830189, 0.32578616]), 'test_f1_macro': array([0.32501702, 0.28883036, 0.26805155, 0.28212759, 0.30735728,
       0.28757187, 0.28853785, 0.27382272, 0.25892857, 0.25771144])}


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for impu


--- CHOL: voting_clf_soft ---
{'fit_time': array([59.88277888, 60.1219883 , 59.60548115, 60.19642162, 59.85888815,
       59.22886467, 59.04944348, 59.61213589, 59.23854995, 59.31317782]), 'score_time': array([0.32844162, 0.32428741, 0.3368578 , 0.33006263, 0.33643508,
       0.31968856, 0.32544684, 0.32712507, 0.33944774, 0.33139968]), 'test_accuracy': array([0.64563107, 0.64320388, 0.64563107, 0.64563107, 0.64563107,
       0.64563107, 0.64720195, 0.64233577, 0.64476886, 0.64476886]), 'test_precision_macro': array([0.21521036, 0.21703522, 0.21521036, 0.21521036, 0.21573398,
       0.21521036, 0.21573398, 0.21463415, 0.21492295, 0.21492295]), 'test_recall_macro': array([0.33333333, 0.3320802 , 0.33333333, 0.33333333, 0.33333333,
       0.33333333, 0.33333333, 0.33207547, 0.33333333, 0.33333333]), 'test_f1_macro': array([0.26155359, 0.26250619, 0.26155359, 0.26155359, 0.26193993,
       0.26155359, 0.26193993, 0.26074074, 0.26134122, 0.26134122])}


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/p


--- CHOL: stacking_clf ---
{'fit_time': array([264.50445795, 264.29745722, 264.84223771, 265.7114563 ,
       266.22221899, 264.96157289, 264.90167332, 264.78778529,
       265.03111506, 265.68972707]), 'score_time': array([0.32507563, 0.32839775, 0.32783699, 0.32015371, 0.32545352,
       0.31970215, 0.32724237, 0.44861317, 0.45939803, 0.45545053]), 'test_accuracy': array([0.49029126, 0.50485437, 0.43932039, 0.47087379, 0.51213592,
       0.49271845, 0.44525547, 0.45498783, 0.46958637, 0.39902676]), 'test_precision_macro': array([0.47820985, 0.41151717, 0.34607555, 0.27856124, 0.41962351,
       0.48126984, 0.26456061, 0.34412159, 0.28005585, 0.32086086]), 'test_recall_macro': array([0.46645816, 0.43733124, 0.35617789, 0.37740628, 0.43102383,
       0.3981755 , 0.35194369, 0.38157164, 0.38862572, 0.36852778]), 'test_f1_macro': array([0.36299488, 0.36795641, 0.33395324, 0.29474307, 0.36079795,
       0.33841634, 0.27741324, 0.31032136, 0.29799182, 0.29297057])}


In [ ]:
for name in models:
    print(f"\n--- BP: {name} ---")
    print("Accuracy:", bp_results[name]["test_accuracy"].mean())
    print("Precision:", bp_results[name]["test_precision_macro"].mean())
    print("Recall:", bp_results[name]["test_recall_macro"].mean())
    print("F1_macro:", bp_results[name]["test_f1_macro"].mean())

In [ ]:
for name in models:
    print(f"\n--- CHOL: {name} ---")
    print("Accuracy:", chol_results[name]["test_accuracy"].mean())
    print("Precision:", chol_results[name]["test_precision_macro"].mean())
    print("Recall:", chol_results[name]["test_recall_macro"].mean())
    print("F1_macro:", chol_results[name]["test_f1_macro"].mean())
# 05.12.26
# --- CHOL: stump ---
# Accuracy: 0.4922294663737509
# Precision: 0.2804081907378314
# Recall: 0.39279056357272146
# F1_macro: 0.306121482305952

# --- CHOL: tree ---
# Accuracy: 0.5087390451893322
# Precision: 0.35380002660558807
# Recall: 0.35249141167843223
# F1_macro: 0.35254647796114724

# --- CHOL: rf ---
# Accuracy: 0.6389654642949945
# Precision: 0.38867996256532916
# Recall: 0.34533129888668473
# F1_macro: 0.298079102975937

# --- CHOL: ada ---
# Accuracy: 0.6460161103630737
# Precision: 0.2887973551675455
# Recall: 0.33422155677794774
# F1_macro: 0.2636823322925438

# --- CHOL: svm ---
# Accuracy: 0.4834644367278482
# Precision: 0.3805001289098701
# Recall: 0.3871179353060705
# F1_macro: 0.37726242172531543

# --- CHOL: nb ---
# Accuracy: 0.6222054898070063
# Precision: 0.37641949373833616
# Recall: 0.3405433716228027
# F1_macro: 0.3025784504556678

# --- CHOL: logreg ---
# Accuracy: 0.4322183639241254
# Precision: 0.38078349167080106
# Recall: 0.39333996762084494
# F1_macro: 0.35354494877360787

# --- CHOL: knn ---
# Accuracy: 0.6217147379113221
# Precision: 0.36433821903340347
# Recall: 0.33949659244951624
# F1_macro: 0.3008003970073593

# --- CHOL: mlp ---
# Accuracy: 0.5808966999740155
# Precision: 0.3795330304372709
# Recall: 0.3654220930103732
# F1_macro: 0.3612881466785069

# --- CHOL: gb ---
# Accuracy: 0.6452867739116057
# Precision: 0.2487677574670244
# Recall: 0.33342236994102514
# F1_macro: 0.2623143298346896

# --- CHOL: hgb ---
# Accuracy: 0.6455294923582077
# Precision: 0.21517649745273587
# Recall: 0.33333333333333337
# F1_macro: 0.26152851336307964

# --- CHOL: voting_clf ---
# Accuracy: 0.6430958117780454
# Precision: 0.35838073752949967
# Recall: 0.34207485889264355
# F1_macro: 0.2837956262664401

# --- CHOL: voting_clf_soft ---
# Accuracy: 0.6450434649091725
# Precision: 0.21538246559380392
# Recall: 0.33308223388660335
# F1_macro: 0.26160235957696115

# --- CHOL: stacking_clf ---
# Accuracy: 0.4679050622445846
# Precision: 0.36248560850675293
# Recall: 0.3957241728252065
# F1_macro: 0.3237558879988812


--- CHOL: stump ---
Accuracy: 0.4922294663737509
Precision: 0.2804081907378314
Recall: 0.39279056357272146
F1_macro: 0.306121482305952

--- CHOL: tree ---
Accuracy: 0.5087390451893322
Precision: 0.35380002660558807
Recall: 0.35249141167843223
F1_macro: 0.35254647796114724

--- CHOL: rf ---
Accuracy: 0.6389654642949945
Precision: 0.38867996256532916
Recall: 0.34533129888668473
F1_macro: 0.298079102975937

--- CHOL: ada ---
Accuracy: 0.6460161103630737
Precision: 0.2887973551675455
Recall: 0.33422155677794774
F1_macro: 0.2636823322925438

--- CHOL: svm ---
Accuracy: 0.4834644367278482
Precision: 0.3805001289098701
Recall: 0.3871179353060705
F1_macro: 0.37726242172531543

--- CHOL: nb ---
Accuracy: 0.6222054898070063
Precision: 0.37641949373833616
Recall: 0.3405433716228027
F1_macro: 0.3025784504556678

--- CHOL: logreg ---
Accuracy: 0.4322183639241254
Precision: 0.38078349167080106
Recall: 0.39333996762084494
F1_macro: 0.35354494877360787

--- CHOL: knn ---
Accuracy: 0.6217147379113221


grid random search w f1_macro

05/09/26 running cv with new f1 params

In [ ]:
#f1 based params
models = {
    "stump": DecisionTreeClassifier(criterion= 'gini', max_depth=1, random_state=42, class_weight="balanced"),
    "tree": DecisionTreeClassifier(criterion= 'gini', max_depth=None, max_features= None, min_samples_leaf=1, min_samples_split=5, random_state=42, class_weight="balanced"),
    "rf": RandomForestClassifier(criterion='gini', max_depth= 9, max_features= 'sqrt', min_samples_leaf=4, min_samples_split=2, n_estimators=100, random_state=42, class_weight="balanced"),
    "ada": AdaBoostClassifier(n_estimators=200, learning_rate =1.0, random_state=42),
    "svm": SVC(kernel='rbf', gamma='scale', C=1, random_state=42, class_weight="balanced"),
    "nb": GaussianNB(var_smoothing= 1e-09),
    "logreg": LogisticRegression(C= 10, max_iter= 1000, penalty= 'l2', solver='sag', random_state=42, class_weight="balanced"),
    "knn": KNeighborsClassifier(algorithm= 'ball_tree', metric='manhattan', n_neighbors= 3, weights= 'uniform'),
    "mlp": MLPClassifier(solver='adam', max_iter= 500, learning_rate= 'adaptive', hidden_layer_sizes=(100,),alpha= 0.0001, activation='relu', random_state=42),
    "gb": GradientBoostingClassifier(learning_rate= 1.0, max_depth= 3, n_estimators= 200, random_state=42),
    "hgb": HistGradientBoostingClassifier(max_iter= 100, max_depth= 5, learning_rate= 1.0, l2_regularization= 0.5, random_state=42),

    "voting_clf": VotingClassifier(estimators=[
    ('stump',DecisionTreeClassifier(criterion= 'gini', max_depth=1, random_state=42, class_weight="balanced")),
    ('ada', AdaBoostClassifier(n_estimators=200, learning_rate =1.0, random_state=42)),
    ('tree', DecisionTreeClassifier(criterion= 'gini', max_depth=None, max_features= None, min_samples_leaf=1, min_samples_split=5, random_state=42, class_weight="balanced")),
    ('rfc',RandomForestClassifier(criterion='gini', max_depth= 9, max_features= 'sqrt', min_samples_leaf=4, min_samples_split=2, n_estimators=100, random_state=42, class_weight="balanced")),
    ('nb', GaussianNB(var_smoothing= 1e-09)),
    ('lr', LogisticRegression(C= 10, max_iter= 1000, penalty= 'l2', solver='sag', random_state=42, class_weight="balanced")),
    ('knn', KNeighborsClassifier(algorithm= 'ball_2tree', metric='manhattan', n_neighbors= 3, weights= 'uniform')),
    ('svm', SVC(kernel='rbf', gamma='scale', C=1, random_state=42, class_weight="balanced")),
     ('mlp', MLPClassifier(solver='adam', max_iter= 500, learning_rate= 'adaptive', hidden_layer_sizes=(100,),alpha= 0.0001, activation='relu', random_state=42)),
     ('gb', GradientBoostingClassifier(learning_rate= 1.0, max_depth= 3, n_estimators= 200, random_state=42)),
     ('hgb', HistGradientBoostingClassifier(max_iter= 100, max_depth= 5, learning_rate= 1.0, l2_regularization= 0.5, random_state=42)),
    ], voting='hard'),

    "voting_clf_soft": VotingClassifier(estimators=[
        ('ada', AdaBoostClassifier(n_estimators=50, learning_rate = 0.01, random_state=42)),
        ('rfc', RandomForestClassifier(criterion='gini', max_depth= 9, max_features= 'sqrt', min_samples_leaf=4, min_samples_split=2, n_estimators=100, random_state=42, class_weight="balanced")),
        ('nb', GaussianNB(var_smoothing= 1e-09)),
        ('lr', LogisticRegression(C= 10, max_iter= 1000, penalty= 'l2', solver='sag', random_state=42, class_weight="balanced")),
        ('svm',SVC(kernel='rbf', gamma='scale', C=1, random_state=42, probability= True, class_weight="balanced")),
        ('gb', GradientBoostingClassifier(learning_rate= 1.0, max_depth= 3, n_estimators= 200, random_state=42)),
        ('hgb', HistGradientBoostingClassifier(max_iter= 100, max_depth= 5, learning_rate= 1.0, l2_regularization= 0.5, random_state=42)),
    ], voting='soft'),

    "stacking_clf": StackingClassifier(
        estimators=[
            ('ada', AdaBoostClassifier(n_estimators=50, learning_rate = 0.01, random_state=42)),
            ('tree',  DecisionTreeClassifier(criterion= 'gini', max_depth=None, max_features= None, min_samples_leaf=1, min_samples_split=5, random_state=42, class_weight="balanced")),
            ('rfc', RandomForestClassifier(criterion='gini', max_depth= 9, max_features= 'sqrt', min_samples_leaf=4, min_samples_split=2, n_estimators=100, random_state=42, class_weight="balanced")),
            ('nb', GaussianNB(var_smoothing= 1e-09)),
            ('lr', LogisticRegression(C= 10, max_iter= 1000, penalty= 'l2', solver='sag', random_state=42, class_weight="balanced")),
            ('svm', SVC(kernel='rbf', gamma='scale', C=1, random_state=42, class_weight="balanced")),
            ('gb', GradientBoostingClassifier(learning_rate= 1.0, max_depth= 3, n_estimators= 200, random_state=42)),
            ('hgb', HistGradientBoostingClassifier(max_iter= 100, max_depth= 5, learning_rate= 1.0, l2_regularization= 0.5, random_state=42)),
        ],
        final_estimator=LogisticRegression(C= 10, max_iter= 1000, penalty= 'l2', solver='sag', random_state=42, class_weight="balanced")
    )
}

In [ ]:
chol_results = {}

for name in models:
    chol_cv = cross_validate(make_pipeline(plinec,models[name]), chol_training_purple.drop(columns=['CHOL']), chol_training_purple['CHOL'], cv=10, scoring=scoring_list)
    print(f"\n--- CHOL: {name} ---")
    chol_results[name] = chol_cv
    print(chol_results[name])
    print("Accuracy:", chol_results[name]["test_accuracy"].mean())
    print("Precision:", chol_results[name]["test_precision_macro"].mean())
    print("Recall:", chol_results[name]["test_recall_macro"].mean())
    print("F1_macro:", chol_results[name]["test_f1_macro"].mean())


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for impu


--- CHOL: stump ---
{'fit_time': array([0.88934827, 0.66867971, 0.27937317, 0.48501134, 0.25761318,
       0.23778892, 0.31255078, 0.29022479, 0.22480726, 0.4849689 ]), 'score_time': array([0.07881975, 0.08602428, 0.0566113 , 0.05426264, 0.05572248,
       0.05486822, 0.0615871 , 0.05513263, 0.05492878, 0.06290364]), 'test_accuracy': array([0.48786408, 0.4684466 , 0.48786408, 0.52427184, 0.51699029,
       0.45145631, 0.52554745, 0.49878345, 0.50121655, 0.45985401]), 'test_precision_macro': array([0.28314392, 0.27655502, 0.28559177, 0.29356337, 0.27431778,
       0.25916907, 0.27644757, 0.29037205, 0.28974009, 0.27518127]), 'test_recall_macro': array([0.39785634, 0.38199221, 0.41537354, 0.4224924 , 0.36618141,
       0.35260025, 0.37577988, 0.41538873, 0.41081226, 0.38942861]), 'test_f1_macro': array([0.30690728, 0.29509483, 0.31287243, 0.32833722, 0.30233766,
       0.27929057, 0.3088483 , 0.31687508, 0.31612055, 0.29453092])}
Accuracy: 0.4922294663737509
Precision: 0.280408190737831

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/p


--- CHOL: tree ---
{'fit_time': array([0.61350894, 0.50367928, 0.39699936, 0.61598921, 0.52822566,
       0.40957642, 0.40318036, 0.53094888, 0.40101361, 0.55554605]), 'score_time': array([0.04387903, 0.04093289, 0.04315591, 0.0513041 , 0.04118562,
       0.04108167, 0.04049301, 0.04069209, 0.04106236, 0.0405848 ]), 'test_accuracy': array([0.5315534 , 0.45631068, 0.47087379, 0.47330097, 0.52912621,
       0.47330097, 0.48418491, 0.44038929, 0.49878345, 0.47688564]), 'test_precision_macro': array([0.38733333, 0.34745796, 0.32184371, 0.34873714, 0.38762549,
       0.3496284 , 0.3328421 , 0.31534895, 0.36245507, 0.34023462]), 'test_recall_macro': array([0.38718115, 0.35749943, 0.32092212, 0.35348445, 0.38804189,
       0.35219   , 0.33166884, 0.31168233, 0.36520835, 0.34172584]), 'test_f1_macro': array([0.38635735, 0.34528969, 0.32114576, 0.3494494 , 0.3871869 ,
       0.34987414, 0.33173515, 0.31153447, 0.36301659, 0.34007446])}
Accuracy: 0.48347093284199083
Precision: 0.349350677459244

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/p

...


--- CHOL: voting_clf ---
{'fit_time': array([48.1100049 , 47.20575404, 47.8350389 , 47.75360203, 45.86162686,
       48.5490725 , 46.61681581, 48.07554364, 46.57792258, 46.96649241]), 'score_time': array([0.51674771, 0.32833266, 0.31899357, 0.33825707, 0.3317821 ,
       0.4833076 , 0.31721282, 0.32483006, 0.32888079, 0.33734155]), 'test_accuracy': array([0.62378641, 0.60436893, 0.59951456, 0.60194175, 0.62378641,
       0.58980583, 0.60583942, 0.59854015, 0.6107056 , 0.61800487]), 'test_precision_macro': array([0.46565322, 0.36557535, 0.37709596, 0.3867265 , 0.43948152,
       0.38590059, 0.40532247, 0.37007168, 0.39760728, 0.41631501]), 'test_recall_macro': array([0.41799292, 0.36236411, 0.35985784, 0.35899711, 0.39252279,
       0.37506713, 0.38143611, 0.36131312, 0.3685886 , 0.37981258]), 'test_f1_macro': array([0.42245279, 0.34814262, 0.34547548, 0.34615858, 0.38612101,
       0.36281411, 0.3780487 , 0.34430556, 0.3587126 , 0.37176819])}
Accuracy: 0.6076293907826046
Precision: 0.

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/p


--- CHOL: voting_clf_soft ---
{'fit_time': array([41.55309844, 40.3767581 , 42.23641348, 41.4254694 , 39.35201049,
       41.73457098, 40.40594959, 42.1327312 , 38.77566671, 41.20416713]), 'score_time': array([0.23847246, 0.22482276, 0.22412729, 0.33483624, 0.22345161,
       0.23135829, 0.23060417, 0.34088397, 0.23592544, 0.24939251]), 'test_accuracy': array([0.63106796, 0.62621359, 0.62864078, 0.62621359, 0.63592233,
       0.60194175, 0.62043796, 0.63017032, 0.6350365 , 0.63017032]), 'test_precision_macro': array([0.56091667, 0.37005099, 0.37272727, 0.41482066, 0.34131601,
       0.36009646, 0.4270882 , 0.32367719, 0.38434405, 0.46662928]), 'test_recall_macro': array([0.36396789, 0.35189737, 0.35103663, 0.34183056, 0.3431179 ,
       0.33719716, 0.36223918, 0.34055016, 0.3531185 , 0.36648973]), 'test_f1_macro': array([0.34587582, 0.32397222, 0.32008327, 0.30276644, 0.29909001,
       0.30953933, 0.34617835, 0.29663201, 0.32058414, 0.34938857])}
Accuracy: 0.6265815085158151
Precisio

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['HIQ032I']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute


--- CHOL: stacking_clf ---
{'fit_time': array([180.05456543, 173.42529631, 180.98452497, 182.41536355,
       177.47773433, 180.67190051, 179.47468352, 181.59801006,
       178.86356235, 183.22580218]), 'score_time': array([0.2455194 , 0.24381924, 0.24073744, 0.36133838, 0.33672404,
       0.23213124, 0.24225283, 0.22567415, 0.23779655, 0.27464819]), 'test_accuracy': array([0.51941748, 0.31553398, 0.17961165, 0.29854369, 0.31553398,
       0.30097087, 0.26520681, 0.48175182, 0.33576642, 0.18978102]), 'test_precision_macro': array([0.41379225, 0.3862284 , 0.34421694, 0.34410906, 0.40625109,
       0.33820763, 0.60138327, 0.37607791, 0.2608511 , 0.35430649]), 'test_recall_macro': array([0.46397856, 0.34490751, 0.36101779, 0.37168841, 0.37320143,
       0.28813104, 0.3415644 , 0.40869284, 0.30013341, 0.41486618]), 'test_f1_macro': array([0.37552472, 0.27695239, 0.19340917, 0.25753844, 0.30309493,
       0.27203904, 0.21410911, 0.31525554, 0.23682098, 0.18838263])}
Accuracy: 0.32021177332

In [ ]:
for name in models:
    print(f"\n--- BP: {name} ---")
    print("Accuracy:", bp_results[name]["test_accuracy"].mean())
    print("Precision:", bp_results[name]["test_precision_macro"].mean())
    print("Recall:", bp_results[name]["test_recall_macro"].mean())
    print("F1_macro:", bp_results[name]["test_f1_macro"].mean())


--- BP: stump ---
Accuracy: 0.6329468150896723
Precision: 0.6271251969387724
Recall: 0.6224305599360794
F1_macro: 0.621542557325528

--- BP: tree ---
Accuracy: 0.5707854050711194
Precision: 0.5622611469248622
Recall: 0.5618448495552892
F1_macro: 0.5615609210989172

--- BP: rf ---
Accuracy: 0.6519970109255824
Precision: 0.6482829281015783
Recall: 0.6290031750036797
F1_macro: 0.627199665878164

--- BP: ada ---
Accuracy: 0.6315790558647703
Precision: 0.6282856923390281
Recall: 0.6134631342122415
F1_macro: 0.6094010831049727

--- BP: svm ---
Accuracy: 0.6177391259534117
Precision: 0.6097834061961704
Recall: 0.6081415420845687
F1_macro: 0.6082856057901559

--- BP: nb ---
Accuracy: 0.6397412904555763
Precision: 0.6322040910449424
Recall: 0.6244084274269854
F1_macro: 0.6247466533640889

--- BP: logreg ---
Accuracy: 0.6388399299113585
Precision: 0.6345905536410861
Recall: 0.6356306482474401
F1_macro: 0.6342998606159957

--- BP: knn ---
Accuracy: 0.6161616161616161
Precision: 0.606525759885278

In [ ]:
for name in models:
    print(f"\n--- CHOL: {name} ---")
    print("Accuracy:", chol_results[name]["test_accuracy"].mean())
    print("Precision:", chol_results[name]["test_precision_macro"].mean())
    print("Recall:", chol_results[name]["test_recall_macro"].mean())
    print("F1_macro:", chol_results[name]["test_f1_macro"].mean())

    #05.12.26

#     --- CHOL: stump ---
# Accuracy: 0.4922294663737509
# Precision: 0.2804081907378314
# Recall: 0.39279056357272146
# F1_macro: 0.306121482305952

# --- CHOL: tree ---
# Accuracy: 0.48347093284199083
# Precision: 0.34935067745924414
# Recall: 0.35096043921273395
# F1_macro: 0.34856639039278076

# --- CHOL: rf ---
# Accuracy: 0.5476082488838494
# Precision: 0.39111946424929667
# Recall: 0.38617916209958086
# F1_macro: 0.3873745694663552

# --- CHOL: ada ---
# Accuracy: 0.6460161103630737
# Precision: 0.2887973551675455
# Recall: 0.33422155677794774
# F1_macro: 0.2636823322925438

# --- CHOL: svm ---
# Accuracy: 0.45479413223726173
# Precision: 0.3938241030150477
# Recall: 0.406968023303172
# F1_macro: 0.37911361645361763

# --- CHOL: nb ---
# Accuracy: 0.6222054898070063
# Precision: 0.37641949373833616
# Recall: 0.3405433716228027
# F1_macro: 0.3025784504556678

# --- CHOL: logreg ---
# Accuracy: 0.4251748045260199
# Precision: 0.382794277615603
# Recall: 0.39573450368699353
# F1_macro: 0.3556317231481147

# --- CHOL: knn ---
# Accuracy: 0.5568415893038526
# Precision: 0.37712522304481216
# Recall: 0.36798718085473336
# F1_macro: 0.36381091093023765

# --- CHOL: mlp ---
# Accuracy: 0.5624388774714761
# Precision: 0.3809752382800683
# Recall: 0.3728983042439493
# F1_macro: 0.372905660807691

# --- CHOL: gb ---
# Accuracy: 0.570703115772565
# Precision: 0.3737578030385789
# Recall: 0.3621516795035937
# F1_macro: 0.35975597942080706

# --- CHOL: hgb ---
# Accuracy: 0.5847896440129449
# Precision: 0.39733273837159333
# Recall: 0.3744893377089433
# F1_macro: 0.3738244896358295

# --- CHOL: voting_clf ---
# Accuracy: 0.6076293907826046
# Precision: 0.4009749572099877
# Recall: 0.37579523067827203
# F1_macro: 0.36639996328462726

# --- CHOL: voting_clf_soft ---
# Accuracy: 0.6265815085158151
# Precision: 0.40216667843286336
# Recall: 0.3511445076208827
# F1_macro: 0.3214110150823084

# --- CHOL: stacking_clf ---
# Accuracy: 0.3202117733210498
# Precision: 0.38254241273475
# Recall: 0.366818157035194
# F1_macro: 0.2633126925297771


--- CHOL: stump ---
Accuracy: 0.4922294663737509
Precision: 0.2804081907378314
Recall: 0.39279056357272146
F1_macro: 0.306121482305952

--- CHOL: tree ---
Accuracy: 0.48347093284199083
Precision: 0.34935067745924414
Recall: 0.35096043921273395
F1_macro: 0.34856639039278076

--- CHOL: rf ---
Accuracy: 0.5476082488838494
Precision: 0.39111946424929667
Recall: 0.38617916209958086
F1_macro: 0.3873745694663552

--- CHOL: ada ---
Accuracy: 0.6460161103630737
Precision: 0.2887973551675455
Recall: 0.33422155677794774
F1_macro: 0.2636823322925438

--- CHOL: svm ---
Accuracy: 0.45479413223726173
Precision: 0.3938241030150477
Recall: 0.406968023303172
F1_macro: 0.37911361645361763

--- CHOL: nb ---
Accuracy: 0.6222054898070063
Precision: 0.37641949373833616
Recall: 0.3405433716228027
F1_macro: 0.3025784504556678

--- CHOL: logreg ---
Accuracy: 0.4251748045260199
Precision: 0.382794277615603
Recall: 0.39573450368699353
F1_macro: 0.3556317231481147

--- CHOL: knn ---
Accuracy: 0.5568415893038526
P